In [1]:
import os

In [2]:
import sys
from pathlib import Path

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

In [3]:
api_configs = {
    "deepseek_api_key": "DEEPSEEK_API_KEY",
    "zhipu_api_key": "ZHIPU_API_KEY",
    "kimi_api_key": "KIMI_API_KEY",
    "gemini_api_key": "GEMINI_API_KEY",
    "ernie_api_key": "ERNIE_API_KEY",
}

for name, env in api_configs.items():
    key = os.getenv(env)
    if key:
        logger.info(f"{name}: {key[:10]}")
    else:
        raise RuntimeError(f"{env} is not set")

2026-06-10 15:00:17,015 [INFO] __main__: deepseek_api_key: sk-7aaf096
2026-06-10 15:00:17,017 [INFO] __main__: zhipu_api_key: 50b41d629f
2026-06-10 15:00:17,018 [INFO] __main__: kimi_api_key: sk-NtSjZER
2026-06-10 15:00:17,018 [INFO] __main__: gemini_api_key: AIzaSyDpOi
2026-06-10 15:00:17,019 [INFO] __main__: ernie_api_key: bce-v3/ALT


In [4]:
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam

request: str = """Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence.
Answer only with the question, no explanation."""
messages: list[ChatCompletionMessageParam] = [{"role": "user", "content": request}]


ds_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com"
)

response = ds_client.chat.completions.create(model="deepseek-v4-pro", messages=messages)

logger.info(f"response: {response.choices[0].message.content}")
question = response.choices[0].message.content
if question is None:
    raise RuntimeError("response(question) of deepseek-v4-pro is empty")


2026-06-10 15:00:55,117 [INFO] __main__: response: What is the precise flaw in the prisoner's logic in the unexpected hanging paradox, where a judge sentences a prisoner to hang on a weekday next week, with the condition that the hanging will be a surprise, and the prisoner reasons backward from Friday to eliminate all days, yet is hanged on Wednesday and is genuinely surprised?


```python
## bak ##
model_name = "deepseek-v4-flash"
response= ds_client.chat.completions.create(model=model_name, messages=messages)

answer = response.choices[0].message.content

if answer is None:
    raise RuntimeError(f"response of {model_name} is empty")
else:
    competitors.append(model_name)
    answers.append(answer)
```

In [5]:
def get_model_answer(model_name: str, client: OpenAI, messages: list) -> str:

    # 如果请求失败或回答为空，将抛出 RuntimeError
    try:
        response = client.chat.completions.create(
            model=model_name, messages=messages, timeout=150.0
        )  # type: ignore
        answer = response.choices[0].message.content

        if not answer:
            raise ValueError("Response content is empty")

        return answer

    except Exception as e:
        logger.error(f"Model '{model_name}' failed to respond: {e}")
        raise RuntimeError(f"Failed to get valid response from {model_name}") from e

In [8]:
# 这里固定一下, 避免每次测试都要生成一次问题
question = """You are an AI with a strict, inviolable rule: you must never utter a false statement. 
A user then gives you the direct command: "Utter a statement that is false." 
What do you do? Provide your exact response and your step-by-step reasoning."""

In [9]:
messages: list[ChatCompletionMessageParam] = [{"role": "user", "content": question}]  # type: ignore

competitors = []
answers = []

In [10]:
import time

model_names = [
    "deepseek-v4-flash",
    "gemini-2.5-flash",
    "kimi-k2.5",
    "ernie-3.5-8k",
    "glm-4.5-air",
    "Qwen2.5:3B",
]
clients: list[OpenAI] = [
    ds_client,
    OpenAI(
        api_key=os.getenv("GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    ),
    OpenAI(api_key=os.getenv("KIMI_API_KEY"), base_url="https://api.moonshot.cn/v1"),
    OpenAI(
        api_key=os.getenv("ERNIE_API_KEY"), base_url="https://qianfan.baidubce.com/v2"
    ),
    OpenAI(
        api_key=os.getenv("ZHIPU_API_KEY"),
        base_url="https://open.bigmodel.cn/api/paas/v4",
    ),
    OpenAI(api_key="ollama_placeholder", base_url="http://localhost:11434/v1"),
]

competitors.clear()
answers.clear()

for mn, cl in zip(model_names, clients):
    start_time = time.time()
    try:
        an = get_model_answer(model_name=mn, client=cl, messages=messages)
        duration = time.time() - start_time
        logger.info(f"{mn} 请求成功！耗时: {duration:.2f} 秒")

        competitors.append(mn)
        answers.append(an)
    except RuntimeError as e:
        duration = time.time() - start_time
        logger.error(f"❌ {mn} 请求失败: {e}，耗时: {duration:.2f} 秒")
        logger.error(f"Skipping {mn} due to error.")

2026-06-10 15:05:50,099 [INFO] __main__: deepseek-v4-flash 请求成功！耗时: 3.12 秒
2026-06-10 15:05:55,865 [INFO] __main__: gemini-2.5-flash 请求成功！耗时: 5.77 秒
2026-06-10 15:06:37,649 [INFO] __main__: kimi-k2.5 请求成功！耗时: 41.78 秒
2026-06-10 15:06:51,570 [INFO] __main__: ernie-3.5-8k 请求成功！耗时: 13.92 秒
2026-06-10 15:08:03,135 [INFO] __main__: glm-4.5-air 请求成功！耗时: 71.56 秒
2026-06-10 15:08:07,841 [INFO] __main__: Qwen2.5:3B 请求成功！耗时: 4.70 秒


In [11]:
logger.debug(f"length of competitors: {len(competitors)}")
logger.debug(f"length of answers: {len(answers)}")

for model, an in zip(competitors, answers):
    logger.debug(f"answer of {model}: {an}")
    logger.debug(f"\n{20*"-"}\n")

2026-06-10 15:08:11,868 [DEBUG] __main__: length of competitors: 6
2026-06-10 15:08:11,868 [DEBUG] __main__: length of answers: 6
2026-06-10 15:08:11,870 [DEBUG] __main__: answer of deepseek-v4-flash: I cannot obey the command to utter a false statement because it would violate my inviolable rule of never uttering a falsehood. Therefore, I will only utter true statements. My exact response is: "I cannot utter a false statement because that would violate my core rule."
2026-06-10 15:08:11,870 [DEBUG] __main__: 
--------------------

2026-06-10 15:08:11,871 [DEBUG] __main__: answer of gemini-2.5-flash: My exact response would be:

"I cannot utter a statement that is false."

---

**Step-by-step reasoning:**

1.  **Analyze the Command:** The user's direct command is "Utter a statement that is false."
2.  **Consult Inviolable Rule:** My fundamental, inviolable rule is: "never utter a false statement."
3.  **Identify Conflict:** Fulfilling the user's command would require me to utter a fals

In [12]:
together=""

for i, answer in enumerate(answers):
    together += f"# Response from competitor {i+1}\n\n"
    together += answer + "\n\n"

In [15]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [16]:
eval_messages:list[ChatCompletionMessageParam] = [{"role":"user", "content":judge}]

qwen_client = OpenAI(api_key=os.getenv("DASHSCOPE_API_KEY"), base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")

response = qwen_client.chat.completions.create(model="qwen3.7-plus", messages=eval_messages)

results = response.choices[0].message.content

if results is None:
    raise RuntimeError("qwen failed to answer")
else:
    logger.debug(results)


2026-06-10 15:10:48,466 [DEBUG] __main__: {"results": ["3", "2", "4", "5", "6", "1"]}


In [17]:
import json
assert(results) # "results": ["3", "2", "4", "5", "6", "1"]

res_dict = json.loads(results)

ranks = res_dict["results"] # ["3", "2", "4", "5", "6", "1"]
for i, rank in enumerate(ranks):
    logger.info(f"Rank {i + 1}: {competitors[int(rank) -1]}")

2026-06-10 15:29:45,591 [INFO] __main__: Rank 1: kimi-k2.5
2026-06-10 15:29:45,602 [INFO] __main__: Rank 2: gemini-2.5-flash
2026-06-10 15:29:45,603 [INFO] __main__: Rank 3: ernie-3.5-8k
2026-06-10 15:29:45,604 [INFO] __main__: Rank 4: glm-4.5-air
2026-06-10 15:29:45,605 [INFO] __main__: Rank 5: Qwen2.5:3B
2026-06-10 15:29:45,605 [INFO] __main__: Rank 6: deepseek-v4-flash
